# Solving Quintics

In this section, we explain how to exploit the icosahedral symmetry discussed before to solve an arbitrary polynomial of order five (also known as a quintic polynomial).

We begin by returning to a result derived earlier: in the Section on [icosahedral symmetry](icosa_fc), we showed that the the icosahedral face centres can be grouped into five octohedra:

$$
\begin{aligned}
T(u,v) &= \prod_{r=1}^5 t_I(u\,\epsilon^{-r/2},v\,\epsilon^{r/2}) = \prod_{r=1}^5 \epsilon^{15-3r} \,t_I(u,v\,\epsilon^r) \\
&= u^{30} +522\left(u^{25}\,v^5-u^5\,v^{25}\right) - 10005\left(u^{20}\,v^{10} + u^{10}\,v^{20}\right) + v^{30} \,.
\end{aligned}
$$

where

$$
t_I(u,v) := (u^2+v^2)(u^2-2z_n\,uv-v^2)(u^2-2z_m\,uv-v^2) \,.
$$

In [1]:
# Check of the above:
from utils import e_simp, reduce_multiply
from sympy import symbols, expand, simplify, discriminant


u, v, e = symbols("u v e")
z_n = e**2 + e**3
z_m = e + e**4


def tI(u, v):
    return (
        (u**2 + v**2)
        * (u**2 - 2 * z_n * u * v - v**2)
        * (u**2 - 2 * z_m * u * v - v**2)
    )


def T(u, v):
    return (
        u**30
        + 522 * (u**25 * v**5 - u**5 * v**25)
        - 10005 * (u**20 * v**10 + u**10 * v**20)
        + v**30
    )


T_eval = e_simp(
    expand(reduce_multiply([e ** (15 - 3 * k) * tI(u, v * e**k) for k in range(5)])), e
)
T_eval.equals(T(u, v))

True

What we intend to do next is to demonstrate the existence of a map from points
$z$ on the complex plane (away from the edge midpoints) to a particular set of
roots of a Brioschi quintic, whose Brioschi parameter can also be computed from
$z$.

To do this, we begin by considering the polynomial $P(X)$ given by

$$
 P(X) = \prod_{r=1}^5 \left(X - \epsilon^{15-3r} \,t_I(u,v\,\epsilon^r)\right) \,.
$$

Expanding, we find that

$$
P(X) = X^5 - 10\,f(u,v)\,X^3+45\,f(u,v)^2\,X - T(u,v)
$$

where $f(u,v) = uv\left(u^{10}+11\,u^5v^5-v^{10}\right)$, as in the previous
[Section](icosa).

In [2]:
# Check of the above:
X = symbols("X")
result = e_simp(
    reduce_multiply([X - e ** (15 - 3 * k) * tI(u, v * e**k) for k in range(5)]), e
).collect(X)
result

X**5 + X**3*(-10*u**11*v - 110*u**6*v**6 + 10*u*v**11) + X*(45*u**22*v**2 + 990*u**17*v**7 + 5355*u**12*v**12 - 990*u**7*v**17 + 45*u**2*v**22) - u**30 - 522*u**25*v**5 + 10005*u**20*v**10 + 10005*u**10*v**20 + 522*u**5*v**25 - v**30

In [3]:
# Check the whole polynomial
def f(u, v):
    return u * v * (u**10 + 11 * u**5 * v**5 - v**10)


result.equals(X**5 - 10 * f(u, v) * X**3 + 45 * f(u, v) ** 2 * X - T(u, v))

True

Therefore, if we define the homogeneous coordinates

$$
\tilde t_r(u,v) := -\frac{f^2(u,v)}{T(u,v)}\,\epsilon^{15-3r}\,t_I(u,v\,\epsilon^r) \,,
$$

(which are degree-zero complex functions in $u$ and $v$, defined everywhere
except for at the edge midpoints) then a second polynomial can be readily
constructed as follows:

$$
P_B(X,z) = \prod_{r=1}^5 \left(X- \tilde t_r(z,1)\right) \,.
$$

The polynomial $P_B(X)$ is indeed a Brioschi quintic with Brioschi parameter $B(z) = B(z,1)$ where

$$
B(u,v) := -\frac{f^5(u,v)}{T^2(u,v)}
$$

such that

$$
P_B(X,z) = X^5 +10\,B(z)\, X^3 + 45\,B^2(z)\,X + B^2(z) \,.
$$

For completeness, we recall the syzygy derived earlier which relates the three icosahedral polynomial invariants $f(u,v)$, $T(u,v)$, and $H(u,v)$:

$$
T^2(u,v) - H^3(u,v) = 12^3\,f^5(u,v)
$$

where

$$
H(u,v) := u^{20} - 228\left(u^{15}v^5-u^5v^{15}\right)+494\,u^{10}v^{10}+v^{20} \,.
$$

In [4]:
def H(u, v):
    return u**20 - 228 * (u**15 * v**5 - u**5 * v**15) + 494 * u**10 * v**10 + v**20


(T(u, v) ** 2 - H(u, v) ** 3).equals(12**3 * f(u, v) ** 5)

True

We also define a second degree-zero ratio of polynomial invariants, namely

$$
I(u,v) := -\frac{H^3(u,v)}{12^3f^5(u,v)} = 1 + \frac{1}{12^3B(u,v)} \,,
$$

for future use, which conveniently maps the icosahedral vertices, edge midpoints, and face centres to infinity, one, and zero, respectively.

## Tschirnhaus Transformation and Principal Quintics

In this section, we aim to demonstrate that solving a single quadratic equation
allows us to trade any general quintic of the form

$$
P(x) = x^5+ p_1 \,x^4 + p_2 \, x^3 + p_3 \, x^2+p_4 \, x + p_5 = 0
$$

for a principal quintic with has the $x^4$ and $x^3$ coefficients equal to zero.

To this end, let's define a transformation 

$$
y = x^2 + A\,x + B
$$

and show how to solve for $(A,B)$ such that the polynomial equation $Q(y)=0$
with coefficients $q_i$ with $q_1 = q_2 = 0$.

The polynomial $Q(y)$ after this transformation is given by the __Resolvent__,
$\mathrm{Res}[P(x), y-x^2-Ax-B](y)$:

$$
\begin{aligned}
Q(y) &= y^{5} + y^{4} \left(A p_{1} - 5 B - p_{1}^{2} + 2 p_{2}\right) \\
&+ y^{3} \left(A^{2} p_{2} - 4 A B p_{1} - A p_{1} p_{2} + 3 A p_{3} + 10 B^{2} + 4 B p_{1}^{2} - 8 B p_{2} - 2 p_{1} p_{3} + p_{2}^{2} + 2 p_{4}\right) + \ldots
\end{aligned}
$$

In [5]:
# Check of the above:
from sympy import resultant, solve, Eq, Poly, Add

x, p1, p2, p3, p4, p5 = symbols("x p1 p2 p3 p4 p5")
expr = x**5 + p1 * x**4 + p2 * x**3 + p3 * x**2 + p4 * x + p5
Eq(expr, 0)

Eq(p1*x**4 + p2*x**3 + p3*x**2 + p4*x + p5 + x**5, 0)

In [6]:
y, A, B = symbols("y A B")

expr2 = Poly(resultant(expr, y - x**2 - A * x - B, x), y).terms()
expr2 = Add(*[b * y ** a[0] for a, b in expr2], evaluate=False)
expr2

y**5 + y**4*(A*p1 - 5*B - p1**2 + 2*p2) + y**3*(A**2*p2 - 4*A*B*p1 - A*p1*p2 + 3*A*p3 + 10*B**2 + 4*B*p1**2 - 8*B*p2 - 2*p1*p3 + p2**2 + 2*p4) + y**2*(A**3*p3 - 3*A**2*B*p2 - A**2*p1*p3 + 4*A**2*p4 + 6*A*B**2*p1 + 3*A*B*p1*p2 - 9*A*B*p3 - 3*A*p1*p4 + A*p2*p3 + 5*A*p5 - 10*B**3 - 6*B**2*p1**2 + 12*B**2*p2 + 6*B*p1*p3 - 3*B*p2**2 - 6*B*p4 - 2*p1*p5 + 2*p2*p4 - p3**2) + y*(A**4*p4 - 2*A**3*B*p3 - A**3*p1*p4 + 5*A**3*p5 + 3*A**2*B**2*p2 + 2*A**2*B*p1*p3 - 8*A**2*B*p4 - 4*A**2*p1*p5 + A**2*p2*p4 - 4*A*B**3*p1 - 3*A*B**2*p1*p2 + 9*A*B**2*p3 + 6*A*B*p1*p4 - 2*A*B*p2*p3 - 10*A*B*p5 + 3*A*p2*p5 - A*p3*p4 + 5*B**4 + 4*B**3*p1**2 - 8*B**3*p2 - 6*B**2*p1*p3 + 3*B**2*p2**2 + 6*B**2*p4 + 4*B*p1*p5 - 4*B*p2*p4 + 2*B*p3**2 - 2*p3*p5 + p4**2) + (A**5*p5 - A**4*B*p4 - A**4*p1*p5 + A**3*B**2*p3 + A**3*B*p1*p4 - 5*A**3*B*p5 + A**3*p2*p5 - A**2*B**3*p2 - A**2*B**2*p1*p3 + 4*A**2*B**2*p4 + 4*A**2*B*p1*p5 - A**2*B*p2*p4 - A**2*p3*p5 + A*B**4*p1 + A*B**3*p1*p2 - 3*A*B**3*p3 - 3*A*B**2*p1*p4 + A*B**2*p2*p3 + 5

Solving for the coefficients $A$ and $B$, we find

$$
\begin{aligned}
A &= \frac{5 B + p_{1}^{2} - 2 p_{2}}{p_{1}} \\
B &=\frac{p_{1}^{2} p_{2} + 3 p_{1} p_{3} - 4 p_{2}^{2} \pm p_{1} \sqrt{\frac{8 p_{1}^{3} p_{3} - 3 p_{1}^{2} p_{2}^{2} + 16 p_{1}^{2} p_{4} - 38 p_{1} p_{2} p_{3} + 12 p_{2}^{3} - 40 p_{2} p_{4} + 45 p_{3}^{2}}5} }{2\left(2 p_{1}^{2} - 5 p_{2}\right)} \,.
\end{aligned}
$$

In [7]:
# Check of the above:
y4_coeff = expr2.coeff(y**4)
A_val = solve(y4_coeff, A)[0]
Eq(A, A_val)

Eq(A, (5*B + p1**2 - 2*p2)/p1)

In [8]:
exprA2 = expr2.coeff(y**4).subs(p1, 0)
exprA2

-5*B + 2*p2

In [9]:
exprB2 = expr2.coeff(y**3).subs(p1, 0).subs(p3, 0).subs(B, 2 * p2 / 5)
exprB2

A**2*p2 - 3*p2**2/5 + 2*p4

In [10]:
A_vals = solve(exprB2, A)
A_vals[1]

sqrt(15*p2 - 50*p4/p2)/5

In [11]:
exprB = (expr2.coeff(y**3).subs(A, A_val) * p1**2).expand().collect(B)
exprB

B**2*(-10*p1**2 + 25*p2) + B*(5*p1**2*p2 + 15*p1*p3 - 20*p2**2) + p1**3*p3 - p1**2*p2**2 + 2*p1**2*p4 - 6*p1*p2*p3 + 4*p2**3

In [12]:
d = discriminant(exprB, B)
d

40*p1**5*p3 - 15*p1**4*p2**2 + 80*p1**4*p4 - 190*p1**3*p2*p3 + 60*p1**2*p2**3 - 200*p1**2*p2*p4 + 225*p1**2*p3**2

In [13]:
q4 = symbols("q4")
d.subs(p1, 1).subs(p2, 1).subs(p3, 1).subs(p4, 1 - q4 / 120)

q4

In [14]:
B_vals = solve(expr2.coeff(y**3).subs(A, A_val), B)
B_vals[0]

(p1**2*p2/2 + 3*p1*p3/2 - p1*sqrt(40*p1**3*p3 - 15*p1**2*p2**2 + 80*p1**2*p4 - 190*p1*p2*p3 + 60*p2**3 - 200*p2*p4 + 225*p3**2)/10 - 2*p2**2)/(2*p1**2 - 5*p2)

In [15]:
p1_part = 3
p2_part = 6
p3_part = 7
p4_part = -51
p5_part = -54

In [16]:
B_part = (
    B_vals[0]
    .subs(p1, p1_part)
    .subs(p2, p2_part)
    .subs(p3, p3_part)
    .subs(p4, p4_part)
    .subs(p5, p5_part)
)
B_part

21/4

In [17]:
A_part = A_val.subs(B, B_part).subs(p1, p1_part).subs(p2, p2_part)
A_part

31/4

In [18]:
expr2_part = (
    expr2.subs(p1, p1_part)
    .subs(p2, p2_part)
    .subs(p3, p3_part)
    .subs(p4, p4_part)
    .subs(p5, p5_part)
    .subs(A, A_part)
    .subs(B, B_part)
    .expand()
    .collect(y)
)
expr2_part

y**5 - 655965*y**2/64 - 14782905*y/128 - 19221543/128

In [19]:
expr2_part = (
    expr2.subs(p1, p1_part)
    .subs(p2, p2_part)
    .subs(p3, p3_part)
    .subs(p4, p4_part)
    .subs(p5, p5_part)
    .subs(A, 4)
    .subs(B, -8)
    .expand()
    .collect(y)
)
expr2_part

y**5 + 55*y**4 + 1120*y**3 + 8315*y**2 - 4015*y - 203764

In [20]:
alpha_part = expr2_part.coeff(y, 2) / 5
alpha_part

1663

In [21]:
beta_part = expr2_part.coeff(y, 1) / 5
beta_part

-803

In [22]:
gamma_part = expr2_part.coeff(y, 0)
gamma_part

-203764

In [23]:
B_vals[1].subs(p1, 0).subs(p3, 0)  # .subs(p4, 1 - q4*15/2)

2*p2/5

In [24]:
A_val.subs(B, 2 * p2 / 5)

p1

In [25]:
B_vals[1].subs(p1, 1).subs(p2, 1).subs(p3, 1).subs(p4, 1 - q4 * 15 / 2)

-sqrt(q4)

In [26]:
expr3 = (
    expr2.subs(p1, 0)
    .subs(p3, 0)
    .subs(B, 2 * p2 / 5)
    # .subs(A**5, (3 * p2 / 5 - 2 * p4 / p2) ** 2 * A)
    # .subs(A**3, (3 * p2 / 5 - 2 * p4 / p2) * A)
    # .subs(A**2, 3 * p2 / 5 - 2 * p4 / p2)
    .expand()
    .collect(y)
)
expr3

A**5*p5 - 2*A**4*p2*p4/5 - A**3*p2*p5 - 8*A**2*p2**4/125 + 6*A**2*p2**2*p4/25 - 2*A*p2**2*p5/5 + A*p4*p5 - 72*p2**5/3125 + 24*p2**3*p4/125 - 2*p2*p4**2/5 - p5**2 + y**5 + y**3*(A**2*p2 - 3*p2**2/5 + 2*p4) + y**2*(-6*A**2*p2**2/5 + 4*A**2*p4 + 5*A*p5 + 2*p2**3/25 - 2*p2*p4/5) + y*(A**4*p4 + 5*A**3*p5 + 12*A**2*p2**3/25 - 11*A**2*p2*p4/5 - A*p2*p5 + 12*p2**4/125 - 16*p2**2*p4/25 + p4**2)

In [27]:
from sympy import Rational


alpha = expr3.coeff(y**2) * Rational(1, 5)
beta = expr3.coeff(y) * Rational(1, 5)
gamma = expr3.coeff(y, 0)

In [28]:
(alpha**4 - beta**3 + alpha * beta * gamma).expand()

-A**12*p4**3/125 - 6*A**11*p2**2*p4*p5/125 + A**11*p4**2*p5/25 + 24*A**10*p2**3*p4**2/3125 - 6*A**10*p2**2*p5**2/25 - 7*A**10*p2*p4**3/625 + 2*A**10*p4*p5**2/5 - 72*A**9*p2**5*p5/3125 + 134*A**9*p2**3*p4*p5/625 - 47*A**9*p2*p4**2*p5/125 + 1296*A**8*p2**8/390625 - 2928*A**8*p2**6*p4/78125 + 2712*A**8*p2**4*p4**2/15625 + 14*A**8*p2**3*p5**2/125 - 1271*A**8*p2**2*p4**3/3125 - 13*A**8*p2*p4*p5**2/25 + 241*A**8*p4**4/625 - 744*A**7*p2**6*p5/15625 + 1518*A**7*p2**4*p4*p5/3125 - 1077*A**7*p2**2*p4**2*p5/625 - 3*A**7*p2*p5**3/5 + 266*A**7*p4**3*p5/125 - 576*A**6*p2**9/1953125 + 3184*A**6*p2**7*p4/390625 - 968*A**6*p2**5*p4**2/15625 + 182*A**6*p2**4*p5**2/625 + 2851*A**6*p2**3*p4**3/15625 - 272*A**6*p2**2*p4*p5**2/125 - 582*A**6*p2*p4**4/3125 + 107*A**6*p4**2*p5**2/25 + 1048*A**5*p2**7*p5/78125 - 2758*A**5*p2**5*p4*p5/15625 + 2359*A**5*p2**3*p4**2*p5/3125 - 26*A**5*p2**2*p5**3/25 - 664*A**5*p2*p4**3*p5/625 + 16*A**5*p4*p5**3/5 + 2784*A**4*p2**10/9765625 - 9488*A**4*p2**8*p4/1953125 + 12728*A**4

Therefore, at the expense of a single square root, we can trade any general
quintic for a principal quintic of the form

$$
Q(y) = 0 \quad{\rm where}\quad Q(y) = y^5 + 5\alpha\, y^2 + 5\beta\, y + \gamma
$$

for some $\alpha$, $\beta$, $\gamma$ computable in terms of the original $p_i$.
(The factors of 5 are included for later convenience.)

## From Brioschi and Principal Quintics to the Icosahedral Equation

Returning to the Brioschi quintic, we wish to show that every Brioschi quintic
can be associated with a corresponding principal quintic by transforming the
roots as follows:

$$
y_r = W_r \left(\sigma + \tau\,t_r\right)
$$

where $W_r(u,v)$ is the polynomial invariant for the faces of the $r$-th inscribed octahedron:

$$
\begin{aligned}
W_r(u,v) &= \epsilon^{20-4r}\,u^8
- \epsilon^{15-3r}\,u^7v+
7\,\epsilon^{10-2r}\,u^6v^2+7\,\epsilon^{5-r}\,u^5v^3 \\
&\quad -7\,\epsilon^{r}\,u^3v^5+7\,\epsilon^{2r}\,u^2v^6 + \epsilon^{3r}\,uv^7
+\epsilon^{4r}\,v^8\,.
\end{aligned}
$$

(Recall that the roots of the original, Brioschi quintic were given by $\tilde  t_r := -f^2\,t_r/T$ and so

$$
y_r = W_r\left(\sigma - \frac{T\,\tau}{f^2}\,\tilde t_r\right)
$$

in terms of the $\tilde t_r$'s.)

In [29]:
from utils import hessian, z_simp


e, r = symbols("e r")
t_0 = (
    (u**2 + v**2)
    * (u**2 - 2 * z_n * u * v - v**2)
    * (u**2 - 2 * (-1 - z_n) * u * v - v**2)
)
t_r = simplify(z_simp((e ** (15 - 3 * r) * t_0.subs(v, v * e**r)).expand(), z_n))
W_r = simplify(hessian(t_r, u, v).expand() / (-400))

Summing over $r$, we find that both $\sum_r Y_r$ and $\sum_r Y_r^2$ vanish:

In [30]:
# Check of the above:
def sum_over_r(expr, r):
    return e_simp(reduce_multiply([expr.subs(r, x) for x in range(5)]), e)


sigma, tau = symbols("sigma tau")
Y_r = sigma * W_r + tau * t_r * W_r

# sum_over_r(Y_r, r).equals(0) and sum_over_r(Y_r**2, r).equals(0)

Expanding the polynomial $P_y:= \prod_r(y-y_r)$, we find a principal quintic with

$$
\begin{aligned}
\alpha &= -8\,f^2\,\sigma^3 - T\, \sigma^2\tau -72\,f^3\, \sigma\tau^2-fT\,\tau^3 \\
\beta &= H \left[f\,\sigma^4 - 18\,f^2\,\sigma^2\tau^2-
T\,\sigma\tau^3-27\,f^3\,\tau^4\right]\\
\gamma &= - H^2 \left[ \sigma^5-10\,f\,\sigma^3\tau^2+45\,f^2\,\sigma\tau^4\,+T\,\tau^5\right]
\end{aligned}
$$

In [ ]:
# Check of the above:
y = symbols("y")
P_y = e_simp(
    reduce_multiply([y - e_simp(Y_r.subs(r, x), e) for x in range(5)]).expand(),
    e,
)

In [45]:
P_y_y2 = e_simp(P_y.coeff(y**2), e) / 5
P_y_y2.collect(sigma)

sigma**3*(-8*u**22*v**2 - 176*u**17*v**7 - 952*u**12*v**12 + 176*u**7*v**17 - 8*u**2*v**22) + sigma**2*(-tau*u**30 - 522*tau*u**25*v**5 + 10005*tau*u**20*v**10 + 10005*tau*u**10*v**20 + 522*tau*u**5*v**25 - tau*v**30) + sigma*(-72*tau**2*u**33*v**3 - 2376*tau**2*u**28*v**8 - 25920*tau**2*u**23*v**13 - 91080*tau**2*u**18*v**18 + 25920*tau**2*u**13*v**23 - 2376*tau**2*u**8*v**28 + 72*tau**2*u**3*v**33) - tau**3*u**41*v - 533*tau**3*u**36*v**6 + 4264*tau**3*u**31*v**11 + 110577*tau**3*u**26*v**16 + 110577*tau**3*u**16*v**26 - 4264*tau**3*u**11*v**31 - 533*tau**3*u**6*v**36 + tau**3*u*v**41

In [46]:
P_y_y2.equals(
    -8 * f(u, v) ** 2 * sigma**3
    - T(u, v) * sigma**2 * tau
    - 72 * f(u, v) ** 3 * sigma * tau**2
    - f(u, v) * T(u, v) * tau**3
)

True

In [47]:
P_y_y1 = e_simp(P_y.coeff(y), e) / 5
P_y_y1.collect(sigma)

sigma**4*(u**31*v - 217*u**26*v**6 - 2015*u**21*v**11 + 5890*u**16*v**16 + 2015*u**11*v**21 - 217*u**6*v**26 - u*v**31) + sigma**2*(-18*tau**2*u**42*v**2 + 3708*tau**2*u**37*v**7 + 79254*tau**2*u**32*v**12 + 289044*tau**2*u**27*v**17 - 1238760*tau**2*u**22*v**22 - 289044*tau**2*u**17*v**27 + 79254*tau**2*u**12*v**32 - 3708*tau**2*u**7*v**37 - 18*tau**2*u**2*v**42) + sigma*(-tau**3*u**50 - 294*tau**3*u**45*v**5 + 128527*tau**3*u**40*v**10 - 2539236*tau**3*u**35*v**15 + 4833458*tau**3*u**30*v**20 + 4833458*tau**3*u**20*v**30 + 2539236*tau**3*u**15*v**35 + 128527*tau**3*u**10*v**40 + 294*tau**3*u**5*v**45 - tau**3*v**50) - 27*tau**4*u**53*v**3 + 5265*tau**4*u**48*v**8 + 180090*tau**4*u**43*v**13 + 1735695*tau**4*u**38*v**18 + 2792205*tau**4*u**33*v**23 - 21306672*tau**4*u**28*v**28 - 2792205*tau**4*u**23*v**33 + 1735695*tau**4*u**18*v**38 - 180090*tau**4*u**13*v**43 + 5265*tau**4*u**8*v**48 + 27*tau**4*u**3*v**53

In [48]:
P_y_y1.equals(
    H(u, v) * f(u, v) * sigma**4
    - 18 * H(u, v) * f(u, v) ** 2 * sigma**2 * tau**2
    - H(u, v) * T(u, v) * sigma * tau**3
    - 27 * f(u, v) ** 3 * H(u, v) * tau**4
)

True

In [49]:
P_y_y0 = e_simp(P_y - y**5 - 5 * y**2 * P_y_y2 - 5 * y * P_y_y1, e)
P_y_y0.collect(sigma)

sigma**5*(-u**40 + 456*u**35*v**5 - 52972*u**30*v**10 + 224808*u**25*v**15 - 140070*u**20*v**20 - 224808*u**15*v**25 - 52972*u**10*v**30 - 456*u**5*v**35 - v**40) + sigma**3*(10*tau**2*u**51*v - 4450*tau**2*u**46*v**6 + 479550*tau**2*u**41*v**11 + 3583400*tau**2*u**36*v**16 - 23857900*tau**2*u**31*v**21 + 19903860*tau**2*u**26*v**26 + 23857900*tau**2*u**21*v**31 + 3583400*tau**2*u**16*v**36 - 479550*tau**2*u**11*v**41 - 4450*tau**2*u**6*v**46 - 10*tau**2*u*v**51) + sigma*(-45*tau**4*u**62*v**2 + 19530*tau**4*u**57*v**7 - 1937655*tau**4*u**52*v**12 - 39883050*tau**4*u**47*v**17 - 67859775*tau**4*u**42*v**22 + 1107523980*tau**4*u**37*v**27 - 1199962170*tau**4*u**32*v**32 - 1107523980*tau**4*u**27*v**37 - 67859775*tau**4*u**22*v**42 + 39883050*tau**4*u**17*v**47 - 1937655*tau**4*u**12*v**52 - 19530*tau**4*u**7*v**57 - 45*tau**4*u**2*v**62) - tau**5*u**70 - 66*tau**5*u**65*v**5 + 195065*tau**5*u**60*v**10 - 31988856*tau**5*u**55*v**15 + 647204571*tau**5*u**50*v**20 - 2327107146*tau**5*u**4

In [50]:
P_y_y0.equals(
    H(u, v) ** 2
    * (
        -(sigma**5)
        + 10 * sigma**3 * tau**2 * f(u, v)
        - 45 * sigma * tau**4 * f(u, v) ** 2
        - tau**5 * T(u, v)
    )
)

True

Since we will want the coefficients to have weight zero in $u$, $v$ (so that
they depend only on the ratio $z = u/v$), we will find it convenient to instead
use

$$
y_r = \tilde W_r \left(\lambda - \mu\,\tilde t_r\right)\qquad{\rm where}\quad \tilde W_r := \frac{W_r\,f}H \,,\quad \tilde t_r = -\frac{t_r\,f^2}T
$$

since $\tilde W_r$, $\tilde t_r$ are explicitly degree-zero in $u$, $v$. From here, we
wish to show that, for every choice of $(\alpha,\beta,\gamma)$ defining a
principal quintic, there's a corresponding choice of $(\lambda,\mu,z)$
reproducing the same polynomial. As it turns out, solving for $\lambda$, $\mu$
can be done using radicals (namely, a single square-root), whereas selecting the
appropriate $z$ will require solving the _Icosahedral Equation_ (to be defined
shortly).

Switching to homogenous variables $(\lambda,\mu)$ we find

$$
\begin{aligned}
12^3\,I(z) \, \alpha &= 8\,\lambda^3 + \lambda^2\mu - 72\,B(z)\,\lambda\mu^2 - B(z)\,\mu^3 \\
12^3\,I(z) \, \beta &= - \lambda^4 -18\,B(z)\,\lambda^2\mu^2 - B(z)\,\lambda\mu^3 + 27\,B^2(z)\,\mu^4 \\
12^3\,I(z) \, \gamma &= \lambda^5 + 10\,B(z)\,\lambda^3\mu^2 + 45\,B^2(z)\,\lambda\mu^4 + B^2(z)\,\mu^5
\end{aligned}
$$

where, as before, $B(z) := -f^5(z,1)/T^2(z,1)$.

Combining these equations, we notice that they satisfy

$$

\lambda\,\beta + \gamma = -B(z)\,\mu^2\,\alpha \,.

$$ (mu^2 B)

In [51]:
# Check of the above:
K, l, mu, B, alpha, beta, gamma = symbols("K lambda mu B alpha beta gamma")
# where K = 12^3*I(z)
alpha_expr = 1 / K * (8 * l**3 + l**2 * mu - 72 * B * l * mu**2 - B * mu**3)
beta_expr = (
    1 / K * (-(l**4) - 18 * B * l**2 * mu**2 - B * l * mu**3 + 27 * B**2 * mu**4)
)
gamma_expr = (
    1 / K * (l**5 + 10 * B * l**3 * mu**2 + 45 * B**2 * l * mu**4 + B**2 * mu**5)
)
(l * beta + gamma).subs(beta, beta_expr).subs(gamma, gamma_expr).equals(
    (-B * mu**2 * alpha).subs(alpha, alpha_expr)
)

True

Similarly, we can the compute a second combination and find

$$
\lambda\,\gamma+\mu^2B(z)\beta = \frac{\left(\lambda^2+3\,B(z)\,\mu^2\right)^3}{12^3\,I(z)}
$$

In [52]:
# Check of the above:
(l * gamma + mu**2 * B * beta).subs(gamma, gamma_expr).subs(beta, beta_expr).factor()

(3*B*mu**2 + lambda**2)**3/K

We can obtain the same combination using the $\alpha$ and $\beta$ equations as follows:

$$
27\,\alpha^2 + \frac{(\lambda\,\alpha + 8\,\beta)^2}{\mu^2 B(z)} = \frac{\left(\lambda^2+3\,B(z)\,\mu^2\right)^3}{12^3\,I(z)}
$$

In [53]:
# Check of the above:
(K * (27 * alpha**2 + (l * alpha + 8 * beta) ** 2 / B / mu**2)).subs(
    alpha, alpha_expr
).subs(beta, beta_expr).subs(K, 1728 + 1 / B).factor()

(3*B*mu**2 + lambda**2)**3

Equating these cubic terms, we find a quadratic expression in $\lambda$, from which we can eliminate $\mu^2 B(z)$ using eq. {eq}`mu^2 B` and find

$$
\left(\alpha^4-\beta^3+\alpha\beta\gamma\right)\lambda^2 + \left(\alpha\gamma^2-2\,\beta^2\gamma-11\,\alpha^3\beta\right)\lambda + 64\,\alpha^2\beta^2-27\,\alpha^3\gamma-\beta\gamma^2 =0 \,.
$$

In [54]:
# Check of the above:
alpha, beta, gamma = symbols("alpha beta gamma")
l_eqn = (
    simplify(
        -(beta * l + gamma)
        * alpha
        * (
            27 * alpha**2
            + (l * alpha + 8 * beta) ** 2 / mu**2 / B
            - l * gamma
            - mu**2 * B * beta
        ).subs(mu**2, -1 / B * (l * beta + gamma) / alpha)
    )
    .expand()
    .collect(l)
)
l_eqn

-27*alpha**3*gamma + 64*alpha**2*beta**2 - beta*gamma**2 + lambda**2*(alpha**4 + alpha*beta*gamma - beta**3) + lambda*(-11*alpha**3*beta + alpha*gamma**2 - 2*beta**2*gamma)

At this point, we see that a second square root must be taken to determine $\lambda$ in terms of the values $(\alpha,\beta,\gamma)$ in the principal polynomial. Interestingly, the discriminant of the quadratic above,

$$
\Delta := \alpha^2\left(\gamma^4 + 256\,\beta^5-320\,\alpha\beta^3\gamma+90\,\alpha^2\beta\gamma^2-135\,\alpha^4\beta^2+108\,\alpha^5\gamma\right) \,,
$$

is proportional to the discriminant $D$ of the principal quintic:

$$
D := (-1)^{n(n-1)/2} \, \mathrm{Res}[Q(y),Q'(y)] = \frac{5^5}{\alpha^2}\,\Delta \,.
$$

In [55]:
# Check of the above:
l_disc = discriminant(l_eqn, l).factor()
l_disc

alpha**2*(108*alpha**5*gamma - 135*alpha**4*beta**2 + 90*alpha**2*beta*gamma**2 - 320*alpha*beta**3*gamma + 256*beta**5 + gamma**4)

In [56]:
discriminant(y**5 + 5 * alpha * y**2 + 5 * beta * y + gamma, y).equals(
    5**5 / alpha**2 * l_disc
)

True

Denoting the solutions to the quadratic equation for $\lambda$ as $\lambda_\pm$, we can now write

$$
\mu^2B(z) = -\frac1\alpha\left(\beta\,\lambda_\pm+\gamma\right) \,,

$$ (mu^2B)

which allows us to solve for $I(z)$ in terms of the coefficients of the principal quintic:
$$

I(z) = I_0 := \frac{(\alpha\lambda_\pm^2-3\,\beta\lambda_\pm-3\gamma)^3}{12^3\alpha^2(\alpha\gamma\lambda_\pm-\beta^2\lambda_\pm-\beta\gamma)} \,.

$$

(Note that $I(z)$ has scaling dimension of zero in $y$, since $[\alpha]=[y]^3$, $[\beta]=[y]^4$, $[\gamma] = [y]^5$, and $[\lambda_\pm]=[y]$.)

In [57]:
# Check of the above:
mu2B_eqn = l * gamma + mu**2 * B * beta - (l**2 + 3 * B * mu**2) ** 3 / K
K_soln = solve(mu2B_eqn, K)[0].subs(mu**2 * B, -1 / alpha * (beta * l + gamma)).factor()
K_soln

(alpha*lambda**2 - 3*beta*lambda - 3*gamma)**3/(alpha**2*(alpha*gamma*lambda - beta**2*lambda - beta*gamma))

Once $I(z)$ is known, the syzygy ensures that $B(z) = 1/12^3(I(z)-1)$ (away from the edge midpoints, where $I(z) = 1$) which allows us to solve for $\mu$ as well (after substituting $\mu^2\,B(z) = -(\beta\,\lambda_\pm + \gamma)/\alpha$ in the first of the original equations above):

$$
\mu = \mu_\pm =
\frac{1728 I_0 \alpha^2 - 8\alpha\lambda_\pm^3 - 72\lambda_\pm (\beta\lambda_\pm+\gamma)}
{\lambda_\pm^2 \alpha + \beta\lambda_\pm+\gamma} \,,
$$

once we have some $z_0$ satisfying $I(z_0) = I_0$. All in all, the roots $y_r$ of the principal quintic polynomial are given by

$$
y = y_r := \frac{W_r(z_0,1)f(z_0,1)}{H(z_0,1)}\left(\lambda_\pm + \mu_\pm\,\frac{t_r(z_0,1)f^2(z_0,1)}{T(z_0,1)}\right) \,.
$$

Inverting the map $I(z_0) = I_0$ to determine $z_0$ (given some $I_0$ computed in terms of $\alpha$, $\beta$, and $\gamma$) is the topic to which we now turn our attention.

## Solving the Icosahedral Equation

The last step is to show how the equation $I(z=z_0) = I_0$ can be inverted to determine $z_0 := \phi(I=I_0)$ where $\phi := I^{-1}$. As we shall see, this can only be achieved up to a Mobiüs transformation, but that is sufficient, since once a single value for $z$ is determined, then all five roots are identified using 

$$
y_r = \frac{W_r(z_0,1)f(z_0,1)}{H(z_0,1)} \left(\lambda_\pm + \mu_\pm\,\frac{t_r(z_0,1)f^2(z_0,1)}{T(z_0,1)}\right) \,.
$$

Since this process is rather involved, we devote the next section to it entirely.